# Testing new graph.

In [1]:
import math
from typing import List
from collections import defaultdict

import networkx as nx
import numpy as np
import gensim
import spacy
import nltk
import glob
import random
from pathlib import Path
import tqdm
import torch
from src.data.preprocessing import preprocessing_imdb

DEP_TO_IGNORE = {"PUNCT", "ADP"}
# DEP_TO_IGNORE = {"punct", "det", "aux", "cc", "mark", "case"}

glove_vectors = gensim.models.KeyedVectors.load("../data/external/embeddings/glove_legal_100.bin", mmap='r')
mean_vec = np.mean(glove_vectors.vectors, axis=0)


def get_embedding(token: str):
    token = token.lower().strip()
    if token in glove_vectors:
        return torch.tensor(glove_vectors[token])
    elif "<unk>" in glove_vectors:
        return torch.tensor(glove_vectors["<unk>"])
    else:
        return torch.tensor(mean_vec)


def preprocess_node(target: str):
    return target.lower().strip()


def is_valid_head_and_child(head, child, child_dep, pmi):
    if not head or not child or head == child:  return False
    if child_dep in DEP_TO_IGNORE:  return False
    if (head, child) not in pmi: return False
    return True


def compute_pmi(corpus, window_size=2):
    co_occur, word_count = defaultdict(int), defaultdict(int)
    total_edges = 0

    for text, label in tqdm.tqdm(corpus, "Calculating PMI"):
        doc = nlp(text)

        for token in doc:
            head = preprocess_node(token.head.text)
            child = preprocess_node(token.text)

            if not head or not child or head == child:
                continue

            if token.dep_ in DEP_TO_IGNORE:
                continue

            co_occur[(head, child)] += 1
            word_count[head] += 1
            word_count[child] += 1
            total_edges += 1

    pmi = {}
    for (w1, w2), join_count in co_occur.items():
        p_w1 = word_count[w1] / total_edges
        p_w2 = word_count[w2] / total_edges
        p_w1_w2 = join_count / total_edges
        pmi_value = math.log2(p_w1_w2 / (p_w1 * p_w2))

        pmi[(w1, w2)] = max(pmi_value, 0.0)

    return pmi


def text_to_graph(text: str, pmi: dict, spacy_nlp: spacy.Language, min_pmi: float = 0.0):
    """
    Build a token‑occurrence‑level dependency graph with only dependency edges.
    - Each token occurrence is a unique node.
    - Only dependency edges (head → child) weighted by PMI.
    - Sentence roots may be connected sequentially to preserve discourse structure.
    """
    doc = spacy_nlp(text)
    G = nx.MultiDiGraph()
    node_id_map = {}

    # Create nodes
    for token in doc:
        node_id = f"{token.lemma_}_{token.i}"
        node_id_map[token] = node_id

        if token.dep_ in DEP_TO_IGNORE: continue
        token_text = preprocess_node(token.text)

        G.add_node(
            node_id,
            lemma=token.lemma_,
            pos=token.pos_,
            text=token.text,
            index=token.i,
            sentence_id=token.sent.start,
            # for visualization: ensure label shows lemma
            label=token.lemma_
        )

    # Create dependency edges
    for token in doc:
        head = preprocess_node(token.head.text)
        child = preprocess_node(token.text)

        if token.dep_ in DEP_TO_IGNORE:
            continue

        # fix is_valid check: pass token.dep_ as well
        if not is_valid_head_and_child(head, child, token.dep_, pmi):
            continue

        head_id = node_id_map[token.head]
        child_id = node_id_map[token]

        pmi_val = pmi.get((head, child), 0.0)
        if pmi_val < min_pmi:            continue

        G.add_edge(
            child_id,head_id,
            weight=pmi_val + 1,
            dep=token.dep_,
            label=token.dep_
        )

    # Connect sentence roots to preserve discourse structure
    prev_root_id = None
    for sent in doc.sents:
        root_id = node_id_map[sent.root]
        if prev_root_id is not None:
            G.add_edge(
                prev_root_id, root_id,
                weight=1.0,
                label="sent_seq"
            )
            G.add_edge(
                root_id, prev_root_id,
                weight=1.0,
                label="sent_seq"
            )
        prev_root_id = root_id

    return G


nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

nlp = spacy.load('en_core_web_lg')

# Carregar os documentos
DATASET_PATH = "../data/datasets/IMDB/train/raw/@SENTIMENT/*.txt"
positive_imdb = glob.glob(DATASET_PATH.replace("@SENTIMENT", "positive"))
negative_imdb = glob.glob(DATASET_PATH.replace("@SENTIMENT", "negative"))

positive_raw_dataset = [open(f).read() for f in tqdm.tqdm(positive_imdb, desc="Reading positive reviews")]
negative_raw_dataset = [open(f).read() for f in tqdm.tqdm(negative_imdb, desc="Reading negative reviews")]

# Option 2: List of tuples (review_text, label)
dataset = [(review, "positive") for review in positive_raw_dataset] + \
          [(review, "negative") for review in negative_raw_dataset]

random.seed(40)
random.shuffle(dataset)

dataset = dataset[:5000]
dataset = [(preprocessing_imdb(text, nlp=nlp), label) for text, label in dataset]
dataset[:5]

/home/trdp/Software/anaconda3/envs/phd_env/lib/python3.10/site-packages/spacy/util.py:910: UserWarning: [W095] Model 'en_core_web_lg' (3.8.0) was trained with spaCy v3.8.0 and may not be 100% compatible with the current version (3.7.0). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)
Reading negative reviews: 100%|██████████| 20000/20000 [00:03<00:00, 6550.71it/s]
/media/trdp/STORAGE/3.Trabalho/PhD-Research/ThesisExperiments/3.ToValidate/TextualHGNN/src/data/preprocessing.py:436: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  text_content = BeautifulSoup(raw_text, "html.parser").get_text()


[('Laughably awful. One might think that all the big-name actors involved in this movie would at least make it believable, but they do not--this one is a stinker.Characters either sprint around without having a good reason to know where they are going, or they stand around making constipated faces when they should be running for their lives.Check your higher brain functions at the door if you intend to try watching this movie. Or, get a bunch of your most clever friends together and give this one the "Mystery Science Theater 3000" treatment. That\'s what I was doing through the second half of this turkey.And PLEASE don\'t confuse this with the excellent movie "The Day of the Jackal," a far-superior thriller from 30 years ago.',
  'negative'),
 ('Well, I had to be generous and give this a 2. This was mainly due to the gratuitous holes cut in that lady\'s shirt where her breasts are. I found that mildly amusing. Other than that, this movie does nothing more than provide a few good laughs

In [2]:
pmi_calculation = compute_pmi(dataset)

Calculating PMI: 100%|██████████| 5000/5000 [04:37<00:00, 18.04it/s]


In [3]:
from src.multi_graph.visualization import visualize_structural_graph, visualization_l0

for idx, (text, label) in enumerate(dataset[:5]):
    # Build graph for the given text
    G = text_to_graph(
        text=text,
        pmi=pmi_calculation,
        spacy_nlp=nlp,
        min_pmi=0.0
    )

    # Optionally map dependency labels to numeric ids (if you need for something else)
    deps = nlp.get_pipe("parser").labels
    dep_label_map = {dep: idx2 for idx2, dep in enumerate(deps)}

    # Create an output filename that includes index + label to avoid overwrite
    output_file = f"test_result_{idx}_{label}.html"

    # Visualize with PyVis
    visualization_l0(
        nx_graph=G,
        output_filename=output_file
    )

    print(f"Saved visualization for item {idx} (label={label}) → {output_file}")


Saved visualization for item 0 (label=negative) → test_result_0_negative.html
Saved visualization for item 1 (label=negative) → test_result_1_negative.html
Saved visualization for item 2 (label=positive) → test_result_2_positive.html
Saved visualization for item 3 (label=positive) → test_result_3_positive.html
Saved visualization for item 4 (label=positive) → test_result_4_positive.html
